In [90]:
import os
import pandas as pd
import random

current_folder = os.getcwd()
project_folder = os.path.dirname(current_folder)

csv_file = os.path.join(project_folder, "Data", "EloRatings_clean.csv")

df = pd.read_csv(csv_file)

df["club"] = df["club"].replace({"FC Kobenhavn": "FC Copenhagen", "Steaua": "FCSB"})
df_1750 = (df[(df["elo"] > 1750) & (df["country"] != "RUS") & (df["club"] != "Recreativo")]
    .sort_values(by="elo", ascending=False)
    .drop_duplicates(subset="club"))

All_teams_1750 = df_1750["club"].tolist()
Selected_teams = random.sample(All_teams_1750, 36)


clubs = list(sorted(df["club"].unique())) 
clubs_country = df.set_index("club")["country"].to_dict()
sorted_clubs_country = dict(sorted(clubs_country.items(), key = lambda item: item[1]))


def country_code():
    Club_code = {}
    for team in Selected_teams:
        Club_code[team] = clubs_country[team]
    return Club_code


def initialize_games():
    teams = {}
    for team in Selected_teams:
        teams[team] = {"Matches": [] }
    return teams 

def generate_all_games():
    all_games = []
    for teamA in Selected_teams:
        for teamB in Selected_teams:
            if teamA != teamB:
                home_team = teamA
                away_team = teamB
                game = (home_team, away_team)
                all_games.append(game)
    return all_games
    

def shuffle_games():
    all_games = generate_all_games()
    random.shuffle(all_games)
    return all_games

def is_valid_game(home_team, away_team, teams):
    game = (home_team, away_team)
    game_switch = (away_team, home_team)
    Club_code = country_code()
    if game in teams[home_team]["Matches"]:
        return False
    elif game in teams[away_team]["Matches"]:
        return False
    elif home_team == away_team:
        return False
    elif game_switch in teams[home_team]["Matches"]:
        return False
    elif game_switch in teams[away_team]["Matches"]:
        return False 
    elif Club_code[home_team] == Club_code[away_team]:
        return False 
    elif len(teams[home_team]["Matches"]) > 7:
        return False
    elif len(teams[away_team]["Matches"]) > 7:
        return False
    else:
        return True 
    
def all_teams_complete(teams):
    for team in teams:
        if len(teams[team]["Matches"]) < 8:
            return False
    return True


def valid_games():
    while True: 
        teams = initialize_games()
        all_games = shuffle_games()
        for game in all_games:
            home_team, away_team = game
            if is_valid_game(home_team, away_team, teams): 
                teams[home_team]["Matches"].append(game) 
                teams[away_team]["Matches"].append(game)
                if all_teams_complete(teams):
                    return teams
                 
        


                 
            



def print_table(teams):
    print(f"{'Team':<25} {'Wedstrijden'}")
    print("-" * 70)

    for team in teams:
        print(f"{team:<25} {len(teams[team]['Matches'])}  {teams[team]['Matches']}")

print(Selected_teams)
teams = valid_games()
print_table(teams)
print(len(Selected_teams))          # moet 36 zijn
print(len(set(Selected_teams)))  


['Osasuna', 'Reading', 'Milan', 'Nottm Forest', 'Roma', 'Bordeaux', 'Wolves', 'Lille', 'Feyenoord', 'Celtic', 'Auxerre', 'Leicester', 'Malaga', 'Aston Villa', 'Rangers', 'Southampton', 'Alaves', 'Fulham', 'Everton', 'Lyon', 'Panathinaikos', 'Bournemouth', 'Monaco', 'Man United', 'Dortmund', 'Twente', 'Leeds', 'Real Madrid', 'Nice', 'Napoli', 'Hoffenheim', 'Sevilla', 'Schalke 04', 'Celta', 'PSV Eindhoven', 'Girona']
Team                      Wedstrijden
----------------------------------------------------------------------
Osasuna                   8  [('Monaco', 'Osasuna'), ('Osasuna', 'Feyenoord'), ('Everton', 'Osasuna'), ('Osasuna', 'Man United'), ('Bournemouth', 'Osasuna'), ('Leeds', 'Osasuna'), ('Osasuna', 'Nottm Forest'), ('Osasuna', 'PSV Eindhoven')]
Reading                   8  [('Reading', 'Dortmund'), ('Panathinaikos', 'Reading'), ('Nice', 'Reading'), ('Twente', 'Reading'), ('Reading', 'Malaga'), ('Feyenoord', 'Reading'), ('Auxerre', 'Reading'), ('Sevilla', 'Reading')]
Milan  

In [91]:
import math
import importlib
import importlib

from scipy.stats import poisson 
import numpy as np 
import pandas as pd
import sys
import os

project_folder = os.path.dirname(os.getcwd())
sys.path.append(os.path.join(project_folder, "src"))
from champions_league_poules import All_teams_1750
from datapart import Parameter

current_folder = os.getcwd()
project_folder = os.path.dirname(current_folder)

csv_file1 = os.path.join(project_folder, "data", "Matches_clean.csv")
csv_file2 = os.path.join(project_folder, "data", "EloRatings_clean.csv")
df1 = pd.read_csv(csv_file1)
df2 = pd.read_csv(csv_file2)


table = {}
def champions_league_table(): 
    for team in Selected_teams:
        table[team] = {"P": 0, "Pts": 0, "W": 0, "D": 0, "L": 0, "GS": 0, "GC": 0, "GD": 0}
    return table


def simulate_games():
    unique_matches = set()
    games = valid_games()
    for game in games: 
        unique_matches.update(games[game]["Matches"])
    unique_matches = list(unique_matches)
    return unique_matches

def simulation(): # This is the real simulation
    matches = simulate_games()
    table = champions_league_table()
    for hometeam, awayteam in matches: 
        lambda_home, lambda_away = Parameter(hometeam, awayteam)
        home_goals = poisson.rvs(lambda_home)
        away_goals = poisson.rvs(lambda_away)
        update_table(table, hometeam, awayteam, home_goals, away_goals)
    return table

def update_table(table, hometeam, awayteam, home_goals, away_goals):
    table[hometeam]["P"] += 1
    table[awayteam]["P"] += 1
    table[hometeam]["GS"] += home_goals
    table[hometeam]["GC"] += away_goals
    table[awayteam]["GS"] += away_goals
    table[awayteam]["GC"] += home_goals
    A = home_goals - away_goals
    table[hometeam]["GD"] += A
    table[awayteam]["GD"] += -A

    if home_goals > away_goals:
                table[hometeam]["W"] += 1
                table[awayteam]["L"] += 1
                table[hometeam]["Pts"] += 3
    elif away_goals > home_goals:
                table[awayteam]["W"] += 1
                table[hometeam]["L"] += 1
                table[awayteam]["Pts"] += 3
    else: 
                table[awayteam]["D"] += 1
                table[hometeam]["D"] += 1
                table[hometeam]["Pts"] += 1
                table[awayteam]["Pts"] += 1
    return table

simulation()

df_table = pd.DataFrame(table).T
df_table = df_table.sort_values(
    by=["Pts", "GD", "GS"],
    ascending=False
)

print(df_table)



               P  Pts  W  D  L  GS  GC  GD
Feyenoord      8   19  6  1  1  14   4  10
Napoli         8   16  4  4  0  12   5   7
Nice           8   16  4  4  0   8   2   6
PSV Eindhoven  8   14  4  2  2   8   3   5
Roma           8   14  3  5  0   6   2   4
Dortmund       8   14  3  5  0   6   2   4
Milan          8   14  4  2  2   9   6   3
Girona         8   14  4  2  2   6   4   2
Man United     8   13  3  4  1   7   4   3
Malaga         8   13  3  4  1   6   3   3
Alaves         8   13  3  4  1   6   4   2
Lyon           8   13  4  1  3   6   6   0
Fulham         8   12  3  3  2   7   5   2
Nottm Forest   8   12  3  3  2   6   4   2
Monaco         8   12  3  3  2   7   6   1
Leicester      8   12  3  3  2   8   8   0
Celta          8   12  3  3  2   6   7  -1
Real Madrid    8   11  2  5  1   8   5   3
Rangers        8   11  2  5  1   3   2   1
Celtic         8   10  2  4  2   6   3   3
Wolves         8   10  2  4  2   4   4   0
Osasuna        8   10  3  1  4   8  10  -2
Everton    